# Final Submission — Gemma 4 12B Claim Verification

This notebook is the cleaned submission version of the successful final-training notebook.

It contains:

1. exact environment and data-integrity checks,
2. one-pass reconstruction of the audited 935-row real training set,
3. the **original successful final-training cell with its historical output retained**,
4. a frozen-checkpoint loading path that verifies the selected adapter by SHA-256,
5. the official 500-row test → `submission.csv` pipeline.

## Selected frozen checkpoint

**Model:** `google/gemma-4-12B-it`  
**Adapter:** `FINAL_GEMMA4_12B_ALL935_PLUS150`  
**Selected adapter SHA-256:**

`76630ec4620ff7244f3b6c9ef0350617939d33a5bc6f0e9c545816175b646d8e`

The frozen adapter is the checkpoint selected during development. The final test pipeline verifies this exact SHA before inference.

## 1. Environment

The successful final run used:

- `transformers==5.10.1`
- `peft==0.19.1`
- `bitsandbytes==0.50.1`
- FP16 compute
- 4-bit NF4 QLoRA
- NVIDIA T4

On a fresh Kaggle session, run the installation cell once. If Kaggle replaces already-imported packages, restart the session once and continue from the environment-check cell.

In [ ]:
import sys
import subprocess

PACKAGES = [
    "transformers==5.10.1",
    "peft==0.19.1",
    "bitsandbytes==0.50.1",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--no-cache-dir",
        *PACKAGES,
    ]
)

print("Environment packages installed.")

In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import shutil
import hashlib
import platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import peft
import bitsandbytes

from huggingface_hub import login
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Gemma4UnifiedForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model

assert transformers.__version__ == "5.10.1"
assert peft.__version__ == "0.19.1"
assert bitsandbytes.__version__ == "0.50.1"

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN under Kaggle > Add-ons > Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required.")

torch.cuda.set_device(0)

SEED = 42

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

seed_everything()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
)

## 2. Training-data provenance

The final training curriculum was:

**935 audited real examples + 150 audited contrastive examples = 1,085 examples**

The following cell locates the original training data by SHA-256, reconstructs the 935-row semantic-clean set, and stages the files at the exact paths expected by the preserved historical training cell.

The frozen 75-row holdout is staged for provenance only; it is not included in final training.

In [ ]:
import unicodedata

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

RAW_TRAIN_SHA = (
    "26ea9a6998815d0f99f45aff2206f435"
    "781cc71bd92638101d7ea08c2e175d3c"
)
SYNTHETIC_150_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)
HOLDOUT_75_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)
CLEAN_935_SHA = (
    "b706dbf0c0b4aab4cbcd07bb89c5d018"
    "f7c41e47a55b757016ef3d07a9713337"
)

LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"]
ALLOWED_LABELS = set(LABELS)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def find_input_by_sha(expected_sha, suffix=".jsonl", max_bytes=20_000_000):
    matches = []
    for path in INPUT_ROOT.rglob(f"*{suffix}"):
        if not path.is_file():
            continue
        if max_bytes is not None and path.stat().st_size > max_bytes:
            continue
        try:
            if sha256_file(path) == expected_sha:
                matches.append(path)
        except OSError:
            pass
    if not matches:
        raise FileNotFoundError(
            f"No Kaggle input matched SHA-256:\n{expected_sha}"
        )
    return sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]

RAW_TRAIN_PATH = find_input_by_sha(RAW_TRAIN_SHA)
SYNTHETIC_150_SOURCE = find_input_by_sha(SYNTHETIC_150_SHA)
HOLDOUT_75_SOURCE = find_input_by_sha(HOLDOUT_75_SHA)

raw_train = read_jsonl(RAW_TRAIN_PATH)
assert len(raw_train) == 1000
assert len({str(row["id"]) for row in raw_train}) == 1000

LABEL_MAP = {
    "SUPPORTS": "SUPPORTS",
    "SUPPORTED": "SUPPORTS",
    "REFUTES": "REFUTES",
    "REFUTED": "REFUTES",
    "NOT_ENOUGH_INFO": "NOT_ENOUGH_INFO",
    "NEI": "NOT_ENOUGH_INFO",
}

CONFLICT_IDS = {
    "gfcc_v5_tr_0911",
    "gfcc_v5_tr_0102",
}

SEMANTIC_CORRECTIONS = {
    "gfcc_v5_tr_0092": "REFUTES",
    "gfcc_v5_tr_0015": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0520": "SUPPORTS",
    "gfcc_v5_tr_0061": "SUPPORTS",
    "gfcc_v5_tr_0106": "SUPPORTS",
    "gfcc_v5_tr_0368": "REFUTES",
    "gfcc_v5_tr_0211": "SUPPORTS",
    "gfcc_v5_tr_0797": "SUPPORTS",
    "gfcc_v5_tr_0270": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0892": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0074": "REFUTES",
    "gfcc_v5_tr_0467": "REFUTES",
    "gfcc_v5_tr_0417": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0637": "SUPPORTS",
    "gfcc_v5_tr_0333": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0556": "SUPPORTS",
    "gfcc_v5_tr_0472": "SUPPORTS",
    "gfcc_v5_tr_0360": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0464": "REFUTES",
    "gfcc_v5_tr_0458": "REFUTES",
    "gfcc_v5_tr_0246": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0822": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0644": "REFUTES",
}

def normalize_text(value):
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip()

def normalize_label(value):
    if not isinstance(value, str) or not value.strip():
        return None
    key = normalize_text(value).upper().replace(" ", "_")
    return LABEL_MAP.get(key)

cleaned = []

for source in raw_train:
    label = normalize_label(source.get("label"))
    if label is None:
        continue

    row_id = str(source["id"])
    if row_id in CONFLICT_IDS:
        continue

    evidence = source.get("evidence", [])
    if isinstance(evidence, str):
        evidence = [evidence]

    normalized_evidence = []
    seen_passages = set()

    for passage in evidence:
        passage = normalize_text(passage)
        if not passage or passage in seen_passages:
            continue
        seen_passages.add(passage)
        normalized_evidence.append(passage)

    cleaned.append({
        "id": row_id,
        "claim": normalize_text(source["claim"]),
        "evidence": normalized_evidence,
        "label": label,
    })

deduplicated = []
seen_examples = set()

for row in cleaned:
    key = (
        row["claim"],
        tuple(row["evidence"]),
        row["label"],
    )
    if key in seen_examples:
        continue
    seen_examples.add(key)
    deduplicated.append(row)

for row in deduplicated:
    if row["id"] in SEMANTIC_CORRECTIONS:
        row["label"] = SEMANTIC_CORRECTIONS[row["id"]]

real_rows = deduplicated

assert len(real_rows) == 935
assert len({row["id"] for row in real_rows}) == 935
assert Counter(row["label"] for row in real_rows) == Counter({
    "SUPPORTS": 348,
    "REFUTES": 311,
    "NOT_ENOUGH_INFO": 276,
})

RECOVERY = WORK_ROOT / "d1_recovery_and_consolidated_audit"
D4_DATA = WORK_ROOT / "d4_contrastive_curriculum_v1"

RECOVERY.mkdir(parents=True, exist_ok=True)
D4_DATA.mkdir(parents=True, exist_ok=True)

REAL935_PATH = RECOVERY / "train_clean_v3_semantic_recovered.jsonl"
SYNTH150_PATH = D4_DATA / "d4_contrastive_train_v1.jsonl"
HOLDOUT75_PATH = D4_DATA / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"

with open(REAL935_PATH, "w", encoding="utf-8") as f:
    for row in real_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

assert sha256_file(REAL935_PATH) == CLEAN_935_SHA

shutil.copy2(SYNTHETIC_150_SOURCE, SYNTH150_PATH)
shutil.copy2(HOLDOUT_75_SOURCE, HOLDOUT75_PATH)

assert sha256_file(SYNTH150_PATH) == SYNTHETIC_150_SHA
assert sha256_file(HOLDOUT75_PATH) == HOLDOUT_75_SHA

print("Training inputs verified and staged.")
print("Real rows:", len(real_rows))
print("Real labels:", dict(Counter(row["label"] for row in real_rows)))
print("Real935 SHA-256:", sha256_file(REAL935_PATH))
print("Synthetic150 SHA-256:", sha256_file(SYNTH150_PATH))
print("Frozen75 SHA-256:", sha256_file(HOLDOUT75_PATH))


## 3. Historical final-production training run

The next cell is the final-production training cell from the successful development notebook.

Its **historical output is intentionally retained** because it records the exact selected epoch-2 adapter SHA:

`76630ec4620ff7244f3b6c9ef0350617939d33a5bc6f0e9c545816175b646d8e`

The retained output also records the training-set size, class distribution, optimizer-step schedule, losses, and packaged checkpoint.

For the actual competition test, retraining is not required: the submission section below loads the frozen selected adapter and verifies its SHA before inference.

In [5]:
# ============================================================
# CELL 7 — FINAL ALL-935 PRODUCTION TRAINING
#
# FROZEN D4 RECIPE
#
# Training:
#   935 semantically-clean REAL examples
# + 150 SAME audited contrastive examples
# = 1085 total
#
# Training constraints:
#
# - Fresh Gemma-4 12B base.
# - Fresh LoRA initialization.
# - NOT continued from D4.
# - Exact D4 recipe.
# - Fixed 2 epochs.
# - NO dev-set model selection.
# - Frozen holdout75 is NOT used.
# - Organizer300 is NOT used.
# - External30 is NOT used.
# - Old synthetic180 is NOT used.
#
# This creates a FINAL PRODUCTION CANDIDATE.
#
# The proven D4 945-trained champion remains preserved
# separately by Cell 6.
# ============================================================

import os
import re
import gc
import json
import math
import time
import random
import shutil
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import transformers
import peft
import bitsandbytes

from transformers import (
    AutoProcessor,
    Gemma4UnifiedForConditionalGeneration,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ============================================================
# 0. ENVIRONMENT
# ============================================================

assert transformers.__version__ == "5.10.1"
assert peft.__version__ == "0.19.1"
assert bitsandbytes.__version__ == "0.50.1"


MODEL_ID = (
    "google/gemma-4-12B-it"
)

SEED = 42

MAX_LENGTH = 256

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

LR = 2e-4
WEIGHT_DECAY = 0.01

GRAD_ACCUM = 16
EPOCHS = 2
WARMUP_RATIO = 0.05


WORK = Path(
    "/kaggle/working"
)

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

D4_DATA = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

RUN_DIR = (
    WORK
    / "FINAL_gemma4_12b_d4recipe_real935_plus_audited150"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REAL935_PATH = (
    RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

SYNTH150_PATH = (
    D4_DATA
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT75_PATH = (
    D4_DATA
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)


EXPECTED_REAL935_SHA = (
    "b706dbf0c0b4aab4cbcd07bb89c5d018"
    "f7c41e47a55b757016ef3d07a9713337"
)

EXPECTED_SYNTH150_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT75_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)


BASE_PROMPT = """Classify the claim using only the supplied evidence.

SUPPORTS: the evidence establishes the claim.
REFUTES: the evidence contradicts the claim.
NOT_ENOUGH_INFO: the evidence neither establishes nor contradicts the specific claim.

End your response exactly as:
FINAL: SUPPORTS
or
FINAL: REFUTES
or
FINAL: NOT_ENOUGH_INFO"""


TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\."
        r"(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\."
        r"(?:gate_proj|up_proj|down_proj)"
    r")"
)


# ============================================================
# 1. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def seed_everything():

    random.seed(SEED)

    np.random.seed(SEED)

    torch.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def evidence_list(row):

    evidence = row["evidence"]

    if isinstance(
        evidence,
        str,
    ):
        return [evidence]

    return list(evidence)


def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"

        for i, passage
        in enumerate(
            evidence_list(row),
            1,
        )
    )

    return (
        BASE_PROMPT
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",

                    "text": build_user_text(
                        row
                    ),
                }
            ],
        }
    ]


# ============================================================
# 2. VERIFY EXACT DATA
# ============================================================

print("=" * 78)
print("CELL 7 — FINAL ALL-935 PRODUCTION TRAINING")
print("=" * 78)


for path in [
    REAL935_PATH,
    SYNTH150_PATH,
    HOLDOUT75_PATH,
]:

    assert path.exists(), path


assert (
    sha256_file(
        REAL935_PATH
    )
    == EXPECTED_REAL935_SHA
)

assert (
    sha256_file(
        SYNTH150_PATH
    )
    == EXPECTED_SYNTH150_SHA
)

assert (
    sha256_file(
        HOLDOUT75_PATH
    )
    == EXPECTED_HOLDOUT75_SHA
)


real_rows = read_jsonl(
    REAL935_PATH
)

synth_rows = read_jsonl(
    SYNTH150_PATH
)

holdout_rows = read_jsonl(
    HOLDOUT75_PATH
)


assert len(real_rows) == 935
assert len(synth_rows) == 150
assert len(holdout_rows) == 75


real_ids = {
    str(row["id"])
    for row in real_rows
}

synth_ids = {
    str(row["id"])
    for row in synth_rows
}

holdout_ids = {
    str(row["id"])
    for row in holdout_rows
}


assert len(real_ids) == 935
assert len(synth_ids) == 150
assert len(holdout_ids) == 75

assert real_ids.isdisjoint(
    synth_ids
)

assert holdout_ids.isdisjoint(
    real_ids | synth_ids
)


train_rows = (
    real_rows
    + synth_rows
)


assert len(
    train_rows
) == 1085


label_counts = Counter(
    row["label"]
    for row in train_rows
)


assert label_counts == Counter({
    "SUPPORTS": 398,
    "REFUTES": 361,
    "NOT_ENOUGH_INFO": 326,
})


print(
    "Real semantic-clean train:",
    len(real_rows),
)

print(
    "Audited synthetic train:",
    len(synth_rows),
)

print(
    "TOTAL FINAL TRAIN:",
    len(train_rows),
)

print(
    "Labels:",
    label_counts,
)

print(
    "\nFrozen holdout rows:",
    len(holdout_rows),
)

print(
    "Frozen holdout used in training: NO"
)


# ============================================================
# 3. CLEAN GPU
#
# Cell 5 may still have D4 loaded.
# ============================================================

print("\n" + "=" * 78)
print("GPU CLEANUP")
print("=" * 78)


for variable_name in [
    "model",
    "base_model",
    "d1_model",
]:

    if variable_name in globals():

        try:
            del globals()[
                variable_name
            ]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


time.sleep(2)


assert torch.cuda.is_available()


torch.cuda.set_device(0)


allocated = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print(
    "GPU0:",
    torch.cuda.get_device_name(0),
)

print(
    "GPU0 allocated:",
    f"{allocated:.3f} GB",
)


assert allocated < 0.5, (
    "GPU did not cleanly unload. "
    "Restart kernel and rerun only Cell 7."
)


# ============================================================
# 4. HF TOKEN
# ============================================================

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )


        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    print(
        "WARNING: HF_TOKEN unavailable; "
        "cached/public download may still work."
    )


# ============================================================
# 5. PROCESSOR
# ============================================================

print("\n" + "=" * 78)
print("PROCESSOR")
print("=" * 78)


processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)


if hasattr(
    processor,
    "tokenizer",
):

    tokenizer = (
        processor.tokenizer
    )

else:

    tokenizer = (
        processor
    )


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


print(
    "Processor:",
    processor.__class__.__name__,
)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__,
)


# ============================================================
# 6. EXACT TRAINING TOKEN CONSTRUCTION
# ============================================================

TURN_CLOSE_TEXT = (
    "<turn|>\n"
)


turn_close_ids = (
    tokenizer(
        TURN_CLOSE_TEXT,
        add_special_tokens=False,
    )[
        "input_ids"
    ]
)


def build_training_sample(
    row,
    source,
):

    prompt_encoded = (
        processor.apply_chat_template(
            user_messages(row),

            tokenize=True,

            return_dict=True,

            return_tensors="pt",

            add_generation_prompt=True,

            enable_thinking=False,
        )
    )


    prompt_ids = (
        prompt_encoded[
            "input_ids"
        ][0]
        .tolist()
    )


    answer_ids = (
        tokenizer(
            "FINAL: "
            + row["label"],

            add_special_tokens=False,
        )[
            "input_ids"
        ]
    )


    full_ids = (
        prompt_ids
        + answer_ids
        + turn_close_ids
    )


    labels = (
        [-100] * len(prompt_ids)
        + answer_ids
        + turn_close_ids
    )


    assert len(full_ids) == len(labels)


    return {
        "id": str(
            row["id"]
        ),

        "source": source,

        "input_ids": torch.tensor(
            full_ids,
            dtype=torch.long,
        ),

        "attention_mask": torch.ones(
            len(full_ids),
            dtype=torch.long,
        ),

        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),

        "prompt_length": len(
            prompt_ids
        ),

        "full_length": len(
            full_ids
        ),
    }


print("\n" + "=" * 78)
print("TOKENIZE FINAL TRAIN1085")
print("=" * 78)


train_samples = []

real_lengths = []
synth_lengths = []


for index, row in enumerate(
    real_rows,
    1,
):

    sample = build_training_sample(
        row,
        "real",
    )

    train_samples.append(
        sample
    )

    real_lengths.append(
        sample[
            "full_length"
        ]
    )


    if index % 250 == 0:

        print(
            f"  real tokenized "
            f"{index}/935"
        )


for index, row in enumerate(
    synth_rows,
    1,
):

    sample = build_training_sample(
        row,
        "synthetic",
    )

    train_samples.append(
        sample
    )

    synth_lengths.append(
        sample[
            "full_length"
        ]
    )


assert len(
    train_samples
) == 1085


print(
    "\nReal min/median/max:",
    min(real_lengths),
    int(
        np.median(
            real_lengths
        )
    ),
    max(real_lengths),
)

print(
    "Synthetic min/median/max:",
    min(synth_lengths),
    int(
        np.median(
            synth_lengths
        )
    ),
    max(synth_lengths),
)


max_length_seen = max(
    max(real_lengths),
    max(synth_lengths),
)


print(
    "Combined max:",
    max_length_seen,
)


assert max_length_seen <= MAX_LENGTH


print(
    "1085/1085 fit within "
    "max_length=256."
)


# ============================================================
# 7. FRESH GEMMA-4 12B
# ============================================================

quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,

        bnb_4bit_quant_type=(
            "nf4"
        ),

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


print("\n" + "=" * 78)
print("LOAD FRESH GEMMA-4 12B")
print("=" * 78)


seed_everything()


base_model = (
    Gemma4UnifiedForConditionalGeneration
    .from_pretrained(
        MODEL_ID,

        token=hf_token,

        quantization_config=(
            quant_config
        ),

        device_map={
            "": 0
        },

        dtype=torch.float16,

        low_cpu_mem_usage=True,
    )
)


for parameter in (
    base_model.parameters()
):

    parameter.requires_grad = False


try:

    base_model.config.use_cache = False

except Exception:

    pass


print(
    "Base GPU allocated:",
    f"{torch.cuda.memory_allocated(0)/1024**3:.3f} GB",
)


# ============================================================
# 8. FRESH EXACT D4 LoRA
# ============================================================

seed_everything()


lora_config = (
    LoraConfig(
        r=LORA_R,

        lora_alpha=(
            LORA_ALPHA
        ),

        lora_dropout=(
            LORA_DROPOUT
        ),

        bias="none",

        task_type="CAUSAL_LM",

        target_modules=(
            TARGET_REGEX
        ),
    )
)


model = get_peft_model(
    base_model,
    lora_config,
)


if hasattr(
    model,
    "enable_input_require_grads",
):

    model.enable_input_require_grads()


try:

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

except TypeError:

    model.gradient_checkpointing_enable()


trainable_params = sum(
    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad
)


assert (
    trainable_params
    == 32_784_384
)


print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)

print(
    "Exact D4 LoRA capacity: PASS"
)


# ============================================================
# 9. FIXED TRAINING GEOMETRY
#
# ceil(1085 / 16) = 68 steps/epoch
# 68 * 2 = 136
# 5% warmup = round(6.8) = 7
# ============================================================

steps_per_epoch = math.ceil(
    len(train_samples)
    / GRAD_ACCUM
)

total_steps = (
    steps_per_epoch
    * EPOCHS
)

warmup_steps = max(
    1,

    round(
        total_steps
        * WARMUP_RATIO
    ),
)


assert steps_per_epoch == 68
assert total_steps == 136
assert warmup_steps == 7


optimizer = torch.optim.AdamW(
    [
        parameter

        for parameter
        in model.parameters()

        if parameter.requires_grad
    ],

    lr=LR,

    weight_decay=(
        WEIGHT_DECAY
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,

        num_warmup_steps=(
            warmup_steps
        ),

        num_training_steps=(
            total_steps
        ),
    )
)


print("\n" + "=" * 78)
print("FINAL TRAINING PLAN")
print("=" * 78)


print(
    "Rows:",
    1085,
)

print(
    "Real:",
    935,
)

print(
    "Synthetic:",
    150,
)

print(
    "Physical batch:",
    1,
)

print(
    "Grad accumulation:",
    16,
)

print(
    "Effective batch:",
    16,
)

print(
    "Steps/epoch:",
    steps_per_epoch,
)

print(
    "Total steps:",
    total_steps,
)

print(
    "Warmup:",
    warmup_steps,
)

print(
    "Epochs:",
    2,
)

print(
    "LR:",
    LR,
)

print(
    "NO validation-based checkpoint selection."
)


# ============================================================
# 10. BATCH HELPER
# ============================================================

def make_batch(sample):

    return {
        "input_ids": (
            sample[
                "input_ids"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),

        "attention_mask": (
            sample[
                "attention_mask"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),

        "labels": (
            sample[
                "labels"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),
    }


# ============================================================
# 11. FIXED 2-EPOCH TRAINING
# ============================================================

history = []

global_step = 0


torch.cuda.reset_peak_memory_stats(
    0
)


training_start = time.time()


print("\n" + "=" * 78)
print("FINAL TRAINING START")
print("=" * 78)


for epoch in range(
    1,
    EPOCHS + 1,
):

    epoch_start = time.time()

    model.train()


    epoch_rng = random.Random(
        SEED + epoch
    )


    order = list(
        range(
            len(train_samples)
        )
    )


    epoch_rng.shuffle(
        order
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    epoch_loss = 0.0

    real_loss_sum = 0.0
    real_count = 0

    synth_loss_sum = 0.0
    synth_count = 0

    recent_loss = 0.0
    recent_count = 0

    optimizer_steps = 0


    for position, sample_index in enumerate(
        order,
        1,
    ):

        sample = (
            train_samples[
                sample_index
            ]
        )


        batch = make_batch(
            sample
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):

            outputs = model(
                **batch
            )

            raw_loss = outputs.loss


        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                f"Non-finite loss "
                f"epoch={epoch} "
                f"id={sample['id']}"
            )


        (
            raw_loss
            / GRAD_ACCUM
        ).backward()


        loss_value = float(
            raw_loss
            .detach()
            .cpu()
        )


        epoch_loss += (
            loss_value
        )


        if sample["source"] == "real":

            real_loss_sum += (
                loss_value
            )

            real_count += 1

        else:

            synth_loss_sum += (
                loss_value
            )

            synth_count += 1


        recent_loss += (
            loss_value
        )

        recent_count += 1


        should_step = (
            position
            % GRAD_ACCUM
            == 0

            or position
            == len(order)
        )


        if should_step:

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )


            optimizer_steps += 1
            global_step += 1


            if (
                optimizer_steps % 5 == 0
                or optimizer_steps
                == steps_per_epoch
            ):

                print(
                    f"Epoch {epoch} | "
                    f"step "
                    f"{optimizer_steps:02d}/"
                    f"{steps_per_epoch} | "
                    f"global "
                    f"{global_step:03d}/"
                    f"{total_steps} | "
                    f"loss="
                    f"{recent_loss/recent_count:.4f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.8f} | "
                    f"gpu="
                    f"{torch.cuda.memory_allocated(0)/1024**3:.2f}GB | "
                    f"peak="
                    f"{torch.cuda.max_memory_allocated(0)/1024**3:.2f}GB"
                )


                recent_loss = 0.0
                recent_count = 0


        del outputs
        del raw_loss
        del batch


    assert (
        optimizer_steps
        == steps_per_epoch
    )


    avg_loss = (
        epoch_loss
        / len(train_samples)
    )

    avg_real_loss = (
        real_loss_sum
        / real_count
    )

    avg_synth_loss = (
        synth_loss_sum
        / synth_count
    )


    # --------------------------------------------------------
    # SAVE CHECKPOINT
    # --------------------------------------------------------

    adapter_dir = (
        RUN_DIR
        / f"epoch_{epoch}_adapter"
    )


    model.save_pretrained(
        adapter_dir,
        safe_serialization=True,
    )


    processor.save_pretrained(
        adapter_dir
    )


    adapter_file = (
        adapter_dir
        / "adapter_model.safetensors"
    )


    adapter_sha = sha256_file(
        adapter_file
    )


    history.append({
        "epoch": epoch,

        "train_loss": (
            avg_loss
        ),

        "real_train_loss": (
            avg_real_loss
        ),

        "synthetic_train_loss": (
            avg_synth_loss
        ),

        "adapter_sha256": (
            adapter_sha
        ),

        "adapter_dir": str(
            adapter_dir
        ),

        "epoch_seconds": (
            time.time()
            - epoch_start
        ),
    })


    print("\n" + "-" * 78)

    print(
        f"EPOCH {epoch} COMPLETE"
    )

    print("-" * 78)

    print(
        "Overall loss:",
        f"{avg_loss:.4f}",
    )

    print(
        "Real loss:",
        f"{avg_real_loss:.4f}",
    )

    print(
        "Synthetic loss:",
        f"{avg_synth_loss:.4f}",
    )

    print(
        "Adapter SHA:",
        adapter_sha,
    )


# ============================================================
# 12. FINAL EPOCH-2 MANIFEST
# ============================================================

FINAL_ADAPTER_DIR = (
    RUN_DIR
    / "epoch_2_adapter"
)

FINAL_ADAPTER_FILE = (
    FINAL_ADAPTER_DIR
    / "adapter_model.safetensors"
)

FINAL_ADAPTER_SHA = (
    sha256_file(
        FINAL_ADAPTER_FILE
    )
)


SUMMARY_PATH = (
    RUN_DIR
    / "FINAL_ALL935_TRAINING_SUMMARY.json"
)


summary = {
    "experiment": (
        "FINAL_Gemma4_12B_D4_recipe_"
        "real935_plus_audited150"
    ),

    "status": (
        "FINAL_PRODUCTION_CANDIDATE"
    ),

    "base_model": (
        MODEL_ID
    ),

    "training": {
        "real_semantic_clean_rows": 935,

        "real_sha256": (
            EXPECTED_REAL935_SHA
        ),

        "audited_synthetic_rows": 150,

        "synthetic_sha256": (
            EXPECTED_SYNTH150_SHA
        ),

        "total_rows": 1085,

        "labels": dict(
            label_counts
        ),
    },

    "frozen_holdout": {
        "rows": 75,

        "sha256": (
            EXPECTED_HOLDOUT75_SHA
        ),

        "used_for_training": False,

        "used_for_checkpoint_selection": False,
    },

    "recipe": {
        "lora_r": 8,

        "lora_alpha": 16,

        "lora_dropout": 0.05,

        "trainable_parameters": (
            trainable_params
        ),

        "lr": (
            LR
        ),

        "weight_decay": (
            WEIGHT_DECAY
        ),

        "physical_batch": 1,

        "gradient_accumulation": 16,

        "effective_batch": 16,

        "epochs": 2,

        "steps_per_epoch": (
            steps_per_epoch
        ),

        "total_steps": (
            total_steps
        ),

        "warmup_steps": (
            warmup_steps
        ),

        "scheduler": (
            "cosine"
        ),

        "seed": (
            SEED
        ),

        "max_length": (
            MAX_LENGTH
        ),

        "thinking": False,

        "completion_only": True,
    },

    "history": history,

    "final_epoch": 2,

    "final_adapter_sha256": (
        FINAL_ADAPTER_SHA
    ),

    "important_note": (
        "No dev/validation/holdout metric was "
        "used for checkpoint selection. Epoch 2 "
        "was fixed before training."
    ),
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. PACKAGE FINAL PRODUCTION CANDIDATE
# ============================================================

PACKAGE_DIR = (
    WORK
    / "FINAL_GEMMA4_12B_ALL935_PLUS150"
)


if PACKAGE_DIR.exists():

    shutil.rmtree(
        PACKAGE_DIR
    )


PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copytree(
    FINAL_ADAPTER_DIR,
    PACKAGE_DIR
    / "epoch_2_adapter",
)


shutil.copy2(
    SUMMARY_PATH,
    PACKAGE_DIR
    / SUMMARY_PATH.name,
)


manifest_copy = (
    D4_DATA
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


if manifest_copy.exists():

    shutil.copy2(
        manifest_copy,
        PACKAGE_DIR
        / manifest_copy.name,
    )


readme = (
    PACKAGE_DIR
    / "README.txt"
)


readme.write_text(
    f"""FINAL GEMMA-4 12B PRODUCTION CANDIDATE
=========================================

Training:
935 semantically-clean real examples
+ 150 audited contrastive examples
= 1085 examples

Recipe:
Frozen from proven D4 champion.

Fresh base model.
Fresh LoRA initialization.
Two fixed epochs.
No validation-based checkpoint selection.

Final epoch-2 adapter SHA256:
{FINAL_ADAPTER_SHA}

IMPORTANT:
The separately saved D4_GEMMA4_12B_CHAMPION.zip remains
the empirically validated champion.

This all-935 model is the final-production candidate trained
after model/data strategy was frozen.
""",
    encoding="utf-8",
)


ZIP_BASE = (
    WORK
    / "FINAL_GEMMA4_12B_ALL935_PLUS150"
)


existing_zip = Path(
    str(ZIP_BASE)
    + ".zip"
)


if existing_zip.exists():

    existing_zip.unlink()


zip_path = Path(
    shutil.make_archive(
        base_name=str(
            ZIP_BASE
        ),

        format="zip",

        root_dir=str(
            PACKAGE_DIR.parent
        ),

        base_dir=(
            PACKAGE_DIR.name
        ),
    )
)


zip_sha = sha256_file(
    zip_path
)

zip_mb = (
    zip_path.stat().st_size
    / 1024**2
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print("\n" + "=" * 78)
print("CELL 7 COMPLETE — FINAL PRODUCTION CANDIDATE")
print("=" * 78)


print(
    "Final training rows:",
    1085,
)

print(
    "Final epoch:",
    2,
)

print(
    "Final adapter:",
    FINAL_ADAPTER_DIR,
)

print(
    "Final adapter SHA256:",
    FINAL_ADAPTER_SHA,
)


print(
    "\nSummary:",
    SUMMARY_PATH,
)

print(
    "Package ZIP:",
    zip_path,
)

print(
    "ZIP size:",
    f"{zip_mb:.2f} MB",
)

print(
    "ZIP SHA256:",
    zip_sha,
)


print(
    "\nNO validation set was used."
)

print(
    "NO frozen holdout was evaluated."
)

print(
    "NO organizer validation was evaluated."
)

print(
    "NO external30 was evaluated."
)

print(
    "NO old synthetic180 was evaluated."
)


print(
    "\nIMPORTANT:"
)

print(
    "Keep BOTH packages:"
)

print(
    "1. D4_GEMMA4_12B_CHAMPION.zip"
)

print(
    "2. FINAL_GEMMA4_12B_ALL935_PLUS150.zip"
)


print(
    "\nSend me the complete Cell-7 output."
)

CELL 7 — FINAL ALL-935 PRODUCTION TRAINING
Real semantic-clean train: 935
Audited synthetic train: 150
TOTAL FINAL TRAIN: 1085
Labels: Counter({'SUPPORTS': 398, 'REFUTES': 361, 'NOT_ENOUGH_INFO': 326})

Frozen holdout rows: 75
Frozen holdout used in training: NO

GPU CLEANUP
GPU0: Tesla T4
GPU0 allocated: 0.383 GB

PROCESSOR
Processor: Gemma4UnifiedProcessor
Tokenizer: GemmaTokenizer

TOKENIZE FINAL TRAIN1085
  real tokenized 250/935
  real tokenized 500/935
  real tokenized 750/935

Real min/median/max: 173 191 233
Synthetic min/median/max: 192 207 232
Combined max: 233
1085/1085 fit within max_length=256.

LOAD FRESH GEMMA-4 12B


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Base GPU allocated: 7.536 GB
Trainable parameters: 32,784,384
Exact D4 LoRA capacity: PASS

FINAL TRAINING PLAN
Rows: 1085
Real: 935
Synthetic: 150
Physical batch: 1
Grad accumulation: 16
Effective batch: 16
Steps/epoch: 68
Total steps: 136
Warmup: 7
Epochs: 2
LR: 0.0002
NO validation-based checkpoint selection.

FINAL TRAINING START
Epoch 1 | step 05/68 | global 005/136 | loss=2.7273 | lr=0.00014286 | gpu=7.63GB | peak=8.58GB
Epoch 1 | step 10/68 | global 010/136 | loss=0.3210 | lr=0.00019973 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 15/68 | global 015/136 | loss=0.1149 | lr=0.00019811 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 20/68 | global 020/136 | loss=0.1078 | lr=0.00019503 | gpu=7.65GB | peak=8.64GB
Epoch 1 | step 25/68 | global 025/136 | loss=0.0799 | lr=0.00019054 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 30/68 | global 030/136 | loss=0.0683 | lr=0.00018472 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 35/68 | global 035/136 | loss=0.0524 | lr=0.00017764 | gpu=7.63GB | peak=8.65GB

## 4. Load the selected frozen adapter

For official submission, attach the saved final adapter package to Kaggle.

The loader below accepts either:

- an extracted adapter directory somewhere under `/kaggle/input`, or
- the original ZIP package containing the adapter.

It searches for `adapter_model.safetensors`, verifies the exact selected SHA-256, and requires the matching `adapter_config.json`.

This makes the test/submission path independent of whether the historical training cell is rerun.

In [ ]:
import zipfile
from peft import PeftModel

SELECTED_ADAPTER_SHA = (
    "76630ec4620ff7244f3b6c9ef0350617"
    "939d33a5bc6f0e9c545816175b646d8e"
)

MODEL_ID = "google/gemma-4-12B-it"
MAX_LENGTH = 256
MAX_NEW_TOKENS = 24

BASE_PROMPT = """Classify the claim using only the supplied evidence.

SUPPORTS: the evidence establishes the claim.
REFUTES: the evidence contradicts the claim.
NOT_ENOUGH_INFO: the evidence neither establishes nor contradicts the specific claim.

End your response exactly as:
FINAL: SUPPORTS
or
FINAL: REFUTES
or
FINAL: NOT_ENOUGH_INFO"""

def locate_selected_adapter():
    # 1) Prefer an already-extracted adapter.
    for adapter_file in INPUT_ROOT.rglob("adapter_model.safetensors"):
        try:
            if sha256_file(adapter_file) == SELECTED_ADAPTER_SHA:
                adapter_dir = adapter_file.parent
                config = adapter_dir / "adapter_config.json"
                if not config.exists():
                    raise FileNotFoundError(
                        f"Selected weight found but adapter_config.json is missing:\n{adapter_dir}"
                    )
                return adapter_dir
        except OSError:
            pass

    # 2) Otherwise inspect attached ZIP archives without trusting filenames.
    for zip_path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                members = [
                    name for name in zf.namelist()
                    if name.endswith("adapter_model.safetensors")
                ]

                for member in members:
                    h = hashlib.sha256()
                    with zf.open(member, "r") as src:
                        for chunk in iter(lambda: src.read(1024 * 1024), b""):
                            h.update(chunk)

                    if h.hexdigest() != SELECTED_ADAPTER_SHA:
                        continue

                    member_dir = str(Path(member).parent)
                    prefix = member_dir.rstrip("/") + "/"

                    extract_root = Path("/kaggle/working/selected_final_adapter_extract")
                    if extract_root.exists():
                        shutil.rmtree(extract_root)
                    extract_root.mkdir(parents=True, exist_ok=True)

                    relevant = [
                        name for name in zf.namelist()
                        if name.startswith(prefix) and not name.endswith("/")
                    ]

                    for name in relevant:
                        zf.extract(name, extract_root)

                    adapter_dir = extract_root / member_dir
                    config = adapter_dir / "adapter_config.json"

                    if not config.exists():
                        raise FileNotFoundError(
                            "The ZIP contains the selected adapter weights "
                            "but not adapter_config.json in the same adapter directory."
                        )

                    assert sha256_file(
                        adapter_dir / "adapter_model.safetensors"
                    ) == SELECTED_ADAPTER_SHA

                    return adapter_dir

        except zipfile.BadZipFile:
            continue

    raise FileNotFoundError(
        "Selected frozen adapter was not found under /kaggle/input.\n"
        "Attach the saved FINAL_GEMMA4_12B_ALL935_PLUS150 package."
    )

SELECTED_ADAPTER_DIR = locate_selected_adapter()

print("Selected adapter:", SELECTED_ADAPTER_DIR)
print(
    "Adapter SHA-256:",
    sha256_file(SELECTED_ADAPTER_DIR / "adapter_model.safetensors"),
)

# Free any model left in memory by an optional training run.
for name in ["model", "base_model", "processor", "tokenizer"]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.set_device(0)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

tokenizer = (
    processor.tokenizer
    if hasattr(processor, "tokenizer")
    else processor
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Gemma4UnifiedForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    SELECTED_ADAPTER_DIR,
    is_trainable=False,
)

model.eval()
try:
    model.config.use_cache = True
except Exception:
    pass

print("Frozen selected checkpoint loaded.")


## 5. Frozen inference recipe

Inference uses the same evidence-only prompt and deterministic Gemma 4 generation format used during development:

- thinking disabled in the chat template,
- greedy decoding,
- one beam,
- `max_new_tokens = 24`,
- fixed three-label parser.

No test-time training or parameter updates occur.

In [ ]:
FINAL_RE = re.compile(
    r"FINAL\s*:\s*(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.I,
)

LABEL_RE = re.compile(
    r"\b(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)\b",
    flags=re.I,
)

GENERATION_PREFIX_TAIL = (
    "<|turn>model\n<|channel>thought\n<channel|>"
)

def evidence_list(row):
    evidence = row["evidence"]
    if isinstance(evidence, str):
        return [evidence]
    return list(evidence)

def build_user_text(row):
    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(evidence_list(row), 1)
    )

    return (
        BASE_PROMPT
        + "\n\nClaim:\n"
        + str(row["claim"])
        + "\n\nEvidence:\n"
        + evidence_text
    )

def user_messages(row):
    return [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": build_user_text(row),
        }],
    }]

def parse_prediction(text):
    matches = list(FINAL_RE.finditer(text))
    if matches:
        return matches[-1].group(1).upper()

    matches = list(LABEL_RE.finditer(text))
    if matches:
        return matches[-1].group(1).upper()

    return None

def render_inference_prompt(row):
    rendered = processor.apply_chat_template(
        user_messages(row),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    if GENERATION_PREFIX_TAIL not in rendered[-120:]:
        raise RuntimeError("Unexpected Gemma 4 generation prefix.")

    return rendered

@torch.inference_mode()
def predict_rows(rows, description):
    model.eval()

    predictions = []
    raw_outputs = []
    prompt_lengths = []

    for row in tqdm(rows, desc=description):
        prompt_text = render_inference_prompt(row)

        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=False,
        )

        prompt_length = int(inputs["input_ids"].shape[-1])
        prompt_lengths.append(prompt_length)

        if prompt_length > MAX_LENGTH:
            raise ValueError(
                f"{row['id']} has {prompt_length} prompt tokens "
                f"(max={MAX_LENGTH})."
            )

        inputs = {
            key: value.to("cuda:0")
            for key, value in inputs.items()
        }

        generated = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
        )

        new_tokens = generated[0, prompt_length:]

        raw = tokenizer.decode(
            new_tokens,
            skip_special_tokens=False,
        )

        prediction = parse_prediction(raw)

        predictions.append(prediction)
        raw_outputs.append(raw)

        del inputs, generated, new_tokens

    return predictions, raw_outputs, prompt_lengths

print("Frozen inference functions ready.")


## 6. Official 500-row test → submission.csv

Attach the organizer's unlabeled official test JSONL. The final cell:

- finds exactly one valid 500-row unlabeled test set,
- checks unique IDs and required fields,
- runs the frozen selected checkpoint,
- validates every predicted label,
- writes exactly two columns: `id,label`,
- preserves the original test ID order,
- saves UTF-8 `/kaggle/working/submission.csv`.

The generated CSV is the file to submit to the competition.

In [ ]:
def locate_official_test():
    candidates = []

    for path in INPUT_ROOT.rglob("*.jsonl"):
        if not path.is_file():
            continue

        try:
            rows = read_jsonl(path)
        except Exception:
            continue

        if len(rows) != 500:
            continue

        if len({str(row["id"]) for row in rows}) != 500:
            continue

        if any("label" in row for row in rows):
            continue

        if not all(
            {"id", "claim", "evidence"}.issubset(row.keys())
            for row in rows
        ):
            continue

        candidates.append((path, rows))

    if len(candidates) != 1:
        raise RuntimeError(
            "Expected exactly one valid 500-row unlabeled official test JSONL. "
            f"Found: {[str(path) for path, _ in candidates]}"
        )

    return candidates[0]

TEST_PATH, test_rows = locate_official_test()

test_predictions, test_raw, test_lengths = predict_rows(
    test_rows,
    "Official test",
)

assert len(test_predictions) == 500
assert all(
    prediction in ALLOWED_LABELS
    for prediction in test_predictions
)

test_ids = [str(row["id"]) for row in test_rows]

submission = pd.DataFrame({
    "id": test_ids,
    "label": test_predictions,
})

assert list(submission.columns) == ["id", "label"]
assert len(submission) == 500
assert submission["id"].is_unique
assert submission["label"].notna().all()
assert set(submission["label"]).issubset(ALLOWED_LABELS)
assert submission["id"].tolist() == test_ids
assert set(submission["id"]) == set(test_ids)

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
    encoding="utf-8",
)

reloaded = pd.read_csv(
    SUBMISSION_PATH,
    dtype={"id": str},
)

assert list(reloaded.columns) == ["id", "label"]
assert len(reloaded) == 500
assert reloaded["id"].tolist() == test_ids
assert reloaded["id"].is_unique
assert reloaded["label"].isin(LABELS).all()

print("Official test:", TEST_PATH)
print("Selected adapter SHA-256:", SELECTED_ADAPTER_SHA)
print("Submission:", SUBMISSION_PATH)
print("Rows:", len(submission))
print(
    "Prediction distribution:",
    dict(Counter(test_predictions)),
)
print(
    "Prompt tokens (min / median / max):",
    min(test_lengths),
    int(np.median(test_lengths)),
    max(test_lengths),
)
print("Submission SHA-256:", sha256_file(SUBMISSION_PATH))

display(submission.head())
